In [1]:
# ============ 第二步：然后才导入其他库 ============
import random
import numpy as np
import torch
import pandas as pd
import pyterrier as pt
from pyterrier.measures import *
from collections import defaultdict
import ir_datasets
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

In [2]:
from typing import List
import pandas as pd

def get_easy_negatives(
    dataset, 
    qid: str, 
    df_A: pd.DataFrame,
    df_B: pd.DataFrame,
    num_negatives: int = 100,
    strategy: str = 'min'
) -> List[str]:
    """
    从两个系统中提取真正的"简单"负样本（两个系统都认为不相关）
    """
    qrels = dataset.get_qrels()
    
    # 筛选当前qid且label=0的文档
    neg_docs = qrels[(qrels['qid'] == qid) & (qrels['label'] == 0)].copy()
    
    if len(neg_docs) == 0:
        print(f"Warning: qid={qid} 没有找到任何负样本 (label=0)")
        return []
    
    # 获取当前qid的两个系统的结果
    df_A_qid = df_A[df_A['qid'] == qid].set_index('docno')
    df_B_qid = df_B[df_B['qid'] == qid].set_index('docno')
    
    # 为每个负样本计算综合score
    scores = []
    for docno in neg_docs['docno']:
        score_A = df_A_qid.loc[docno, 'score'] if docno in df_A_qid.index else 0
        score_B = df_B_qid.loc[docno, 'score'] if docno in df_B_qid.index else 0
        
        if strategy == 'min':
            combined_score = min(score_A, score_B)
        elif strategy == 'max':
            combined_score = max(score_A, score_B)
        elif strategy == 'mean':
            combined_score = (score_A + score_B) / 2
        else:
            combined_score = min(score_A, score_B)
        
        scores.append(combined_score)
    
    neg_docs['combined_score'] = scores
    
    # 按综合score升序排序（最低的在前）
    neg_docs_sorted = neg_docs.sort_values('combined_score', ascending=True)
    
    # 返回前num_negatives个docno
    return neg_docs_sorted['docno'].head(num_negatives).tolist()


def rank_based_interleaving_dedup_final(
    df_A: pd.DataFrame, 
    df_B: pd.DataFrame, 
    easy_negatives: dict or list,
    target_docs: int = 9,
    smart_easy_negative: bool = True
) -> pd.DataFrame:
    """
    基于文档级去重的交错算法 - 带标签回溯更新
    
    核心逻辑：
    1. 同步扫描 A 和 B 的位置 k
    2. 在位置 k 时，判断文档类型只基于**已扫描过的部分**
    3. 如果文档已被选过，检查是否需要**更新标签**为 Both
    4. 已选过的文档不重复添加（去重）
    """
    all_qids = set(df_A['qid'].unique()) | set(df_B['qid'].unique())
    final_results = []
    
    for qid in sorted(all_qids):
        # 1. 获取并排序
        list_A = df_A[df_A['qid'] == qid].copy()
        list_B = df_B[df_B['qid'] == qid].copy()
        
        list_A = list_A.sort_values('score', ascending=False).reset_index(drop=True)
        list_B = list_B.sort_values('score', ascending=False).reset_index(drop=True)
        
        # 2. 初始化
        query_selection = []
        seen_docs = {}  # 改为字典，存储 {docno: index_in_selection}
        count_both = 0
        count_a_only = 0
        count_b_only = 0
        
        # 3. 同步扫描两个排名
        k = 0
        max_k = max(len(list_A), len(list_B))
        
        while len(query_selection) < target_docs and k < max_k:
            # 获取第k位的文档
            doc_A = list_A.iloc[k]['docno'] if k < len(list_A) else None
            doc_B = list_B.iloc[k]['docno'] if k < len(list_B) else None
            
            # 创建当前已扫描部分的文档集合
            scanned_docs_A = set(list_A.iloc[:k+1]['docno']) if k < len(list_A) else set(list_A['docno'])
            scanned_docs_B = set(list_B.iloc[:k+1]['docno']) if k < len(list_B) else set(list_B['docno'])
            
            # 处理A的第k个文档
            if doc_A is not None:
                if doc_A not in seen_docs:
                    # 首次遇到这个文档
                    if doc_A in scanned_docs_B:
                        label = 'Both'
                        count_both += 1
                    else:
                        label = 'A-Only'
                        count_a_only += 1
                    
                    query_selection.append({
                        'qid': qid,
                        'docno': doc_A,
                        'origin_label': label
                    })
                    seen_docs[doc_A] = len(query_selection) - 1  # 记录位置
                    
                    # 检查是否已满
                    if len(query_selection) >= target_docs:
                        break
                else:
                    # 已经选过，检查是否需要更新标签
                    idx = seen_docs[doc_A]
                    current_label = query_selection[idx]['origin_label']
                    
                    # 如果当前是A-Only，但现在发现在B中也有 → 更新为Both
                    if current_label == 'A-Only' and doc_A in scanned_docs_B:
                        query_selection[idx]['origin_label'] = 'Both'
                        count_a_only -= 1
                        count_both += 1
                    # 如果当前是B-Only，但现在发现在A中也有 → 更新为Both
                    elif current_label == 'B-Only' and doc_A in scanned_docs_A:
                        query_selection[idx]['origin_label'] = 'Both'
                        count_b_only -= 1
                        count_both += 1
            
            # 处理B的第k个文档
            if doc_B is not None:
                if doc_B not in seen_docs:
                    # 首次遇到这个文档
                    if doc_B in scanned_docs_A:
                        label = 'Both'
                        count_both += 1
                    else:
                        label = 'B-Only'
                        count_b_only += 1
                    
                    query_selection.append({
                        'qid': qid,
                        'docno': doc_B,
                        'origin_label': label
                    })
                    seen_docs[doc_B] = len(query_selection) - 1  # 记录位置
                    
                    # 检查是否已满
                    if len(query_selection) >= target_docs:
                        break
                else:
                    # 已经选过，检查是否需要更新标签
                    idx = seen_docs[doc_B]
                    current_label = query_selection[idx]['origin_label']
                    
                    # 如果当前是B-Only，但现在发现在A中也有 → 更新为Both
                    if current_label == 'B-Only' and doc_B in scanned_docs_A:
                        query_selection[idx]['origin_label'] = 'Both'
                        count_b_only -= 1
                        count_both += 1
                    # 如果当前是A-Only，但现在发现在B中也有 → 更新为Both
                    elif current_label == 'A-Only' and doc_B in scanned_docs_B:
                        query_selection[idx]['origin_label'] = 'Both'
                        count_a_only -= 1
                        count_both += 1
            
            k += 1
        
        # 4. 智能Easy-Negative填充
        if smart_easy_negative:
            remaining_slots = target_docs - count_both
            
            if remaining_slots % 2 == 1:
                needed_easy_negatives = 1
                
                if len(query_selection) >= target_docs:
                    # 移除最后一个A-Only或B-Only以保持平衡
                    if count_a_only > count_b_only:
                        for i in range(len(query_selection) - 1, -1, -1):
                            if query_selection[i]['origin_label'] == 'A-Only':
                                removed_doc = query_selection.pop(i)
                                del seen_docs[removed_doc['docno']]
                                count_a_only -= 1
                                break
                    else:
                        for i in range(len(query_selection) - 1, -1, -1):
                            if query_selection[i]['origin_label'] == 'B-Only':
                                removed_doc = query_selection.pop(i)
                                del seen_docs[removed_doc['docno']]
                                count_b_only -= 1
                                break
            else:
                needed_easy_negatives = 0
        else:
            needed_easy_negatives = target_docs - len(query_selection)
        
        # 5. 添加Easy-Negative
        if needed_easy_negatives > 0:
            if isinstance(easy_negatives, dict):
                neg_list = easy_negatives.get(qid, [])
            else:
                neg_list = easy_negatives
            
            added_negatives = 0
            for neg_doc in neg_list:
                if neg_doc not in seen_docs:
                    query_selection.append({
                        'qid': qid,
                        'docno': neg_doc,
                        'origin_label': 'Easy-Negative'
                    })
                    seen_docs[neg_doc] = len(query_selection) - 1
                    added_negatives += 1
                    
                    if added_negatives >= needed_easy_negatives:
                        break
            
            if added_negatives < needed_easy_negatives:
                print(f"Warning: qid={qid} 只填充了 {added_negatives} 个Easy-Negative，"
                      f"还缺少 {needed_easy_negatives - added_negatives} 个")
        
        # 6. 添加排名
        for rank_idx, item in enumerate(query_selection):
            item['rank'] = rank_idx
            final_results.append(item)
    
    result_df = pd.DataFrame(final_results)
    
    if len(result_df) > 0:
        result_df = result_df[['qid', 'docno', 'rank', 'origin_label']]
    
    return result_df


def analyze_interleaving_results(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    分析交错结果，统计各类别的分布
    """
    if len(result_df) == 0:
        return pd.DataFrame()
    
    stats = []
    
    for qid in result_df['qid'].unique():
        qid_data = result_df[result_df['qid'] == qid]
        
        stats.append({
            'qid': qid,
            'total_docs': len(qid_data),
            'A-Only': (qid_data['origin_label'] == 'A-Only').sum(),
            'B-Only': (qid_data['origin_label'] == 'B-Only').sum(),
            'Both': (qid_data['origin_label'] == 'Both').sum(),
            'Easy-Negative': (qid_data['origin_label'] == 'Easy-Negative').sum()
        })
    
    stats_df = pd.DataFrame(stats)
    
    # 添加总计行
    total_row = {
        'qid': 'TOTAL',
        'total_docs': stats_df['total_docs'].sum(),
        'A-Only': stats_df['A-Only'].sum(),
        'B-Only': stats_df['B-Only'].sum(),
        'Both': stats_df['Both'].sum(),
        'Easy-Negative': stats_df['Easy-Negative'].sum()
    }
    
    stats_df = pd.concat([stats_df, pd.DataFrame([total_row])], ignore_index=True)
    
    return stats_df


def analyze_interleaving_results(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    分析交错结果，统计各类别的分布
    
    Args:
        result_df: rank_based_interleaving_dedup函数返回的结果
    
    Returns:
        统计信息的DataFrame
    """
    if len(result_df) == 0:
        return pd.DataFrame()
    
    stats = []
    
    for qid in result_df['qid'].unique():
        qid_data = result_df[result_df['qid'] == qid]
        
        stats.append({
            'qid': qid,
            'total_docs': len(qid_data),
            'A-Only': (qid_data['origin_label'] == 'A-Only').sum(),
            'B-Only': (qid_data['origin_label'] == 'B-Only').sum(),
            'Both': (qid_data['origin_label'] == 'Both').sum(),
            'Easy-Negative': (qid_data['origin_label'] == 'Easy-Negative').sum()
        })
    
    stats_df = pd.DataFrame(stats)
    
    # 添加总计行
    total_row = {
        'qid': 'TOTAL',
        'total_docs': stats_df['total_docs'].sum(),
        'A-Only': stats_df['A-Only'].sum(),
        'B-Only': stats_df['B-Only'].sum(),
        'Both': stats_df['Both'].sum(),
        'Easy-Negative': stats_df['Easy-Negative'].sum()
    }
    
    stats_df = pd.concat([stats_df, pd.DataFrame([total_row])], ignore_index=True)
    
    return stats_df


def add_query_text(result_df: pd.DataFrame, dataset_name: str = "msmarco-passage/trec-dl-2019/judged") -> pd.DataFrame:
    """
    为interleaving结果添加query_text列
    
    Args:
        result_df: rank_based_interleaving_dedup_final的返回结果
        dataset_name: ir_datasets数据集名称
    
    Returns:
        添加了query_text列的DataFrame
    """
    print("\n" + "="*60)
    print("ADDING QUERY TEXT COLUMN")
    print("="*60)
    
    result = result_df.copy()
    result['qid'] = result['qid'].astype(str)
    
    # 加载数据集
    irds_dataset = ir_datasets.load(dataset_name)
    
    # 构建query映射
    print("Building query map...")
    query_map = {}
    for query in irds_dataset.queries_iter():
        query_map[str(query.query_id)] = query.text
    
    print(f"Loaded {len(query_map)} queries")
    
    # 添加query_text列
    result['query_text'] = result['qid'].map(query_map)
    
    # 检查缺失
    missing_queries = result['query_text'].isna().sum()
    print(f"Missing query texts: {missing_queries}")
    
    return result

In [4]:
print("="*60)
print("COMPLETE PIPELINE")
print("="*60)

# 1. 读取去重后的数据
print("\n1. Loading deduplicated data...")
df_colbert = pd.read_csv('df_colbert_deduped.csv')
df_colbert_prf = pd.read_csv('df_colbert_prf_deduped.csv')

df_colbert['qid'] = df_colbert['qid'].astype(str)
df_colbert['docno'] = df_colbert['docno'].astype(str)
df_colbert_prf['qid'] = df_colbert_prf['qid'].astype(str)
df_colbert_prf['docno'] = df_colbert_prf['docno'].astype(str)

print(f"  ColBERT E2E: {len(df_colbert)} rows")
print(f"  ColBERT-PRF: {len(df_colbert_prf)} rows")

# 2. 准备dataset
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')

# 3. 提取 easy negatives
print("\n2. Extracting easy negatives...")
all_qids = set(df_colbert['qid'].unique()) | set(df_colbert_prf['qid'].unique())
easy_negatives = {}

for qid in tqdm(all_qids, desc="Processing queries"):
    easy_negatives[qid] = get_easy_negatives(
        dataset=dataset,
        qid=qid,
        df_A=df_colbert,
        df_B=df_colbert_prf,
        num_negatives=100,
        strategy='min'
    )

# 4. 运行Interleaving
print("\n3. Running interleaving...")
result = rank_based_interleaving_dedup_final(
    df_A=df_colbert,
    df_B=df_colbert_prf,
    easy_negatives=easy_negatives,
    target_docs=9,
    smart_easy_negative=True
)

print(f"  Generated {len(result)} rows")

# 5. 分析结果
print("\n4. Analyzing results...")
stats = analyze_interleaving_results(result)
print("\n" + stats.to_string())

total_stats = stats[stats['qid'] == 'TOTAL'].iloc[0]
b_only_ratio = total_stats['B-Only'] / total_stats['total_docs']
print(f"\nB-Only 比例: {b_only_ratio:.2%}")
print(f"平均每个查询的B-Only数量: {total_stats['B-Only'] / (len(stats) - 1):.2f}")

# 6. 添加passage_text和query_text
print("\n5. Adding text columns...")

# 6.1 从去重后的df中获取passage_text
passage_map_prf = df_colbert_prf.set_index('docno')['passage_text'].to_dict()
passage_map_e2e = df_colbert.set_index('docno')['passage_text'].to_dict()
passage_map = {**passage_map_prf, **passage_map_e2e}

result['passage_text'] = result['docno'].map(passage_map)

# 6.2 检查缺失的passage_text（主要是Easy-Negative）
missing_docnos = result[result['passage_text'].isna()]['docno'].unique()

if len(missing_docnos) > 0:
    print(f"\n⚠️  Found {len(missing_docnos)} documents without passage_text")
    print(f"   These are likely Easy-Negatives not in the original dfs")
    print(f"   Loading these documents from ir_datasets...")
    
    # 加载缺失的文档
    irds_dataset = ir_datasets.load("msmarco-passage/trec-dl-2019/judged")
    missing_doc_map = {}
    
    for doc in tqdm(irds_dataset.docs_iter(), total=8841823, desc="Loading missing docs"):
        doc_id_str = str(doc.doc_id)
        if doc_id_str in missing_docnos:
            missing_doc_map[doc_id_str] = doc.text
            if len(missing_doc_map) == len(missing_docnos):
                print(f"\n✓ Found all {len(missing_docnos)} missing documents!")
                break
    
    # 更新缺失的passage_text
    for idx, row in result[result['passage_text'].isna()].iterrows():
        if row['docno'] in missing_doc_map:
            result.at[idx, 'passage_text'] = missing_doc_map[row['docno']]
    
    print(f"Updated {len(missing_doc_map)} missing passages")

# 6.3 添加query_text
result = add_query_text(result, dataset_name="msmarco-passage/trec-dl-2019/judged")

# 调整列顺序
result = result[['qid', 'docno', 'rank', 'origin_label', 'query_text', 'passage_text']]

# 7. 保存结果
print("\n6. Saving results...")
result.to_csv('result_interleaved_with_text.csv', index=False)
result.to_pickle('result_interleaved_with_text.pkl')

print("\nSaved to:")
print("  - result_interleaved_with_text.csv")
print("  - result_interleaved_with_text.pkl")

# 8. 显示样例
print("\n" + "="*60)
print("SAMPLE RESULTS:")
print("="*60)

for idx, row in result.head(3).iterrows():
    print(f"\n{idx+1}. Query {row['qid']} (Rank {row['rank']}, {row['origin_label']})")
    print(f"   Query: {row['query_text']}")
    print(f"   DocNo: {row['docno']}")
    print(f"   Passage: {row['passage_text'][:120] if pd.notna(row['passage_text']) else 'N/A'}...")

# 9. 数据质量检查
print("\n" + "="*60)
print("DATA QUALITY CHECK:")
print("="*60)
print(f"Total rows: {len(result)}")
print(f"Missing query_text: {result['query_text'].isna().sum()}")
print(f"Missing passage_text: {result['passage_text'].isna().sum()}")
print(f"Complete rows: {len(result.dropna())}")

# 查看缺失的分布
if result['passage_text'].isna().any():
    print("\n⚠️  Missing passage_text by label:")
    missing_by_label = result[result['passage_text'].isna()]['origin_label'].value_counts()
    print(missing_by_label)

print("\n" + "="*60)
print("✅ PIPELINE COMPLETE!")
print("="*60)

COMPLETE PIPELINE

1. Loading deduplicated data...
  ColBERT E2E: 281534 rows
  ColBERT-PRF: 37996 rows

2. Extracting easy negatives...


Processing queries: 100%|██████████| 43/43 [00:01<00:00, 37.08it/s]



3. Running interleaving...
  Generated 387 rows

4. Analyzing results...

        qid  total_docs  A-Only  B-Only  Both  Easy-Negative
0   1037798           9       3       3     3              0
1    104861           9       3       3     2              1
2   1063750           9       3       3     2              1
3   1103812           9       1       1     7              0
4   1106007           9       2       2     4              1
5   1110199           9       3       3     3              0
6   1112341           9       3       3     3              0
7   1113437           9       3       3     2              1
8   1114646           9       2       2     5              0
9   1114819           9       2       2     4              1
10  1115776           9       2       2     4              1
11  1117099           9       2       2     5              0
12  1121402           9       0       0     8              1
13  1121709           9       1       1     7              0
14  112421

Loading missing docs:  79%|███████▉  | 6983822/8841823 [00:26<00:07, 260256.60it/s]



✓ Found all 13 missing documents!
Updated 13 missing passages

ADDING QUERY TEXT COLUMN
Building query map...
Loaded 43 queries
Missing query texts: 0

6. Saving results...

Saved to:
  - result_interleaved_with_text.csv
  - result_interleaved_with_text.pkl

SAMPLE RESULTS:

1. Query 1037798 (Rank 0, Both)
   Query: who is robert gray
   DocNo: 8760871
   Passage: Atlantic Ocean, United States. Robert Gray, (born May 10, 1755, Tiverton, R.I.—died summer 1806, at sea near eastern U.S...

2. Query 1037798 (Rank 1, Both)
   Query: who is robert gray
   DocNo: 7822415
   Passage: Captain Robert Gray carries the flag around the world on his sailing vessel (around the tip of South America, to China, ...

3. Query 1037798 (Rank 2, Both)
   Query: who is robert gray
   DocNo: 8760866
   Passage: Robert Gray was the Democratic candidate for governor of Mississippi in the 2015 elections. Gray won the Democratic prim...

DATA QUALITY CHECK:
Total rows: 387
Missing query_text: 0
Missing passage_t